In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
VERIFIKASI WAVEFORM vs KATALOG (FIXED TIMEZONE)
"""

import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta, timezone
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================

WAVEFORM_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon_juli"
CATALOG_CSV = "/Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_final_output/HYBRID_EARTHQUAKE_CATALOG_2001_2024_FIX.csv"

TIME_TOLERANCE_HOURS = 12  # toleransi 12 jam
OUTPUT_DIR = "verifikasi_waveform_catalog_fixed"
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

# =============================================
# 2. BACA FILE .mseed
# =============================================

print("="*70)
print("🔍 VERIFIKASI WAVEFORM vs KATALOG (FIXED TIMEZONE)")
print("="*70)

print("\n📂 Membaca file .mseed...")
files = list(Path(WAVEFORM_DIR).glob("*.mseed"))
files = [f for f in files if not f.name.startswith('._')]
print(f"✅ Total file .mseed valid: {len(files)}")

# Ekstrak timestamp lengkap
file_data = []
for f in files:
    # Cari pola YYYYMMDD_HHMMSS atau YYYYMMDDHHMMSS
    match = re.search(r'(\d{8})_(\d{6})', f.name) or re.search(r'(\d{14})', f.name)
    if match:
        if len(match.group(1)) == 8 and len(match.group(2)) == 6:
            date_str = match.group(1)
            time_str = match.group(2)
            timestamp_full = f"{date_str}{time_str}"
        else:
            timestamp_full = match.group(1)
        
        try:
            dt_naive = datetime.strptime(timestamp_full, '%Y%m%d%H%M%S')
            # Buat timezone-aware UTC
            dt_utc = dt_naive.replace(tzinfo=timezone.utc)
            file_data.append({
                'file': f.name,
                'path': str(f),
                'datetime': dt_utc,
                'timestamp': timestamp_full,
                'year': dt_utc.year
            })
        except Exception as e:
            pass

df_files = pd.DataFrame(file_data)
print(f"✅ Metadata diekstrak dari {len(df_files)} file")

# =============================================
# 3. BACA KATALOG
# =============================================

print("\n📂 Membaca katalog...")
df_catalog = pd.read_csv(CATALOG_CSV)
df_catalog['datetime'] = pd.to_datetime(df_catalog['time_utc'], utc=True)
df_catalog['year'] = df_catalog['datetime'].dt.year
print(f"✅ Total event di katalog: {len(df_catalog)}")
print(f"   Rentang tahun: {df_catalog['year'].min()} - {df_catalog['year'].max()}")

# =============================================
# 4. COCOKKAN BERDASARKAN WAKTU (TOLERANSI 12 JAM)
# =============================================

print(f"\n📊 Mencocokkan file dengan katalog (toleransi {TIME_TOLERANCE_HOURS} jam)...")

matched = []
unmatched = []

for _, file_row in df_files.iterrows():
    file_dt = file_row['datetime']
    file_year = file_row['year']
    
    # Cari event di katalog dengan waktu dekat
    time_diff = (df_catalog['datetime'] - file_dt).abs()
    mask = time_diff <= pd.Timedelta(hours=TIME_TOLERANCE_HOURS)
    candidates = df_catalog[mask].copy()
    
    if len(candidates) > 0:
        candidates_sorted = candidates.sort_values('magnitude', ascending=False)
        best = candidates_sorted.iloc[0]
        
        matched.append({
            'file': file_row['file'],
            'file_datetime': file_dt,
            'file_year': file_year,
            'event_id': best['event_id'],
            'catalog_datetime': best['datetime'],
            'magnitude': best['magnitude'],
            'depth': best['depth_km'],
            'latitude': best['latitude'],
            'longitude': best['longitude'],
            'source': best['source'],
            'time_diff_hours': (best['datetime'] - file_dt).total_seconds() / 3600
        })
    else:
        unmatched.append({
            'file': file_row['file'],
            'file_datetime': file_dt,
            'file_year': file_year
        })

df_matched = pd.DataFrame(matched) if matched else pd.DataFrame()
df_unmatched = pd.DataFrame(unmatched) if unmatched else pd.DataFrame()

print(f"✅ File tercocokkan: {len(df_matched)}")
print(f"❌ File tidak tercocokkan: {len(df_unmatched)}")
print(f"📊 Success rate: {len(df_matched)/(len(df_matched)+len(df_unmatched))*100:.1f}%")

# =============================================
# 5. STATISTIK
# =============================================

print("\n" + "="*70)
print("📊 STATISTIK KECOCOKAN")
print("="*70)

if len(df_matched) > 0:
    print("\n📡 Distribusi sumber data (matched):")
    for source, count in df_matched['source'].value_counts().items():
        print(f"  {source}: {count} ({count/len(df_matched)*100:.1f}%)")
    
    print(f"\n📊 Magnitudo rata-rata: {df_matched['magnitude'].mean():.2f}")
    print(f"   Min: {df_matched['magnitude'].min():.1f}, Max: {df_matched['magnitude'].max():.1f}")
    
    print(f"\n📊 Tahun file vs katalog:")
    yearly = df_files['year'].value_counts().sort_index()
    for year, count in yearly.items():
        matched_count = len(df_matched[df_matched['file_year'] == year])
        print(f"  {year}: {count} file, {matched_count} matched")

# =============================================
# 6. FILE TIDAK COCOK
# =============================================

if len(df_unmatched) > 0:
    print(f"\n⚠️ {len(df_unmatched)} file tidak tercocokkan:")
    print("  Contoh 10 file pertama:")
    for _, row in df_unmatched.head(10).iterrows():
        print(f"    - {row['file']} (datetime: {row['file_datetime']})")
    if len(df_unmatched) > 10:
        print(f"    ... dan {len(df_unmatched)-10} lainnya")

# =============================================
# 7. SIMPAN HASIL
# =============================================

df_matched.to_csv(f"{OUTPUT_DIR}/matched_files.csv", index=False)
df_unmatched.to_csv(f"{OUTPUT_DIR}/unmatched_files.csv", index=False)
print(f"\n✅ Hasil tersimpan di folder: {OUTPUT_DIR}/")

🔍 VERIFIKASI WAVEFORM vs KATALOG (FIXED TIMEZONE)

📂 Membaca file .mseed...
✅ Total file .mseed valid: 20215
✅ Metadata diekstrak dari 20215 file

📂 Membaca katalog...
✅ Total event di katalog: 232301
   Rentang tahun: 2001 - 2024

📊 Mencocokkan file dengan katalog (toleransi 12 jam)...
✅ File tercocokkan: 20215
❌ File tidak tercocokkan: 0
📊 Success rate: 100.0%

📊 STATISTIK KECOCOKAN

📡 Distribusi sumber data (matched):
  USGS: 14036 (69.4%)
  BMKG: 6179 (30.6%)

📊 Magnitudo rata-rata: 5.35
   Min: 2.2, Max: 9.1

📊 Tahun file vs katalog:
  2001: 1089 file, 1089 matched
  2002: 1242 file, 1242 matched
  2003: 670 file, 670 matched
  2004: 2020 file, 2020 matched
  2005: 2712 file, 2712 matched
  2006: 2479 file, 2479 matched
  2007: 2060 file, 2060 matched
  2008: 1764 file, 1764 matched
  2009: 3858 file, 3858 matched
  2010: 2321 file, 2321 matched

✅ Hasil tersimpan di folder: verifikasi_waveform_catalog_fixed/


In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
MEMERIKSA KEBERADAAN GEMPA MERUSAK INDONESIA DALAM JSON
Mencocokkan event_id atau timestamp dengan daftar gempa merusak.
"""

import json
import re
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================

JSON_PATH = "/Volumes/Extreme SSD/unduhan_waveform_merged/extracted_data.json"

# Daftar gempa merusak Indonesia (2004-2024)
MAJOR_EARTHQUAKES = [
    {"year": 2004, "month": 12, "day": 26, "mag": 9.1, "name": "Aceh 2004"},
    {"year": 2005, "month": 3, "day": 28, "mag": 8.6, "name": "Nias 2005"},
    {"year": 2006, "month": 5, "day": 27, "mag": 6.3, "name": "Yogyakarta 2006"},
    {"year": 2006, "month": 7, "day": 17, "mag": 7.7, "name": "Pangandaran 2006"},
    {"year": 2007, "month": 9, "day": 12, "mag": 8.5, "name": "Bengkulu 2007"},
    {"year": 2009, "month": 9, "day": 30, "mag": 7.6, "name": "Padang 2009"},
    {"year": 2010, "month": 4, "day": 6, "mag": 7.7, "name": "Sumatra 2010"},
    {"year": 2010, "month": 10, "day": 25, "mag": 7.7, "name": "Mentawai 2010"},
    {"year": 2012, "month": 4, "day": 11, "mag": 8.6, "name": "Aceh 2012"},
    {"year": 2013, "month": 4, "day": 6, "mag": 7.2, "name": "Papua 2013"},
    {"year": 2016, "month": 12, "day": 7, "mag": 6.5, "name": "Pidie Jaya 2016"},
    {"year": 2018, "month": 8, "day": 5, "mag": 7.0, "name": "Lombok 2018"},
    {"year": 2018, "month": 9, "day": 28, "mag": 7.5, "name": "Palu 2018"},
    {"year": 2019, "month": 7, "day": 14, "mag": 7.3, "name": "Maluku 2019"},
    {"year": 2021, "month": 1, "day": 14, "mag": 7.0, "name": "Mamuju 2021"},
    {"year": 2022, "month": 11, "day": 21, "mag": 5.6, "name": "Cianjur 2022"},
    {"year": 2023, "month": 1, "day": 18, "mag": 7.2, "name": "Maluku 2023"},
    {"year": 2024, "month": 1, "day": 8, "mag": 7.0, "name": "Maluku 2024"},
]

# =============================================
# 2. LOAD JSON
# =============================================

print("="*70)
print("🔍 MEMERIKSA GEMPA MERUSAK DALAM JSON")
print("="*70)

print(f"\n📂 Membaca JSON: {JSON_PATH}")
with open(JSON_PATH, 'r') as f:
    data = json.load(f)

print(f"✅ Total event dalam JSON: {len(data)}")

# =============================================
# 3. FUNGSI EKSTRAK TANGGAL DARI KEY
# =============================================

def extract_date_from_key(key):
    """
    Ekstrak tanggal (YYYYMMDD) dari key JSON.
    Format key: GE_TNTI_20041226_005853_30
    """
    # Cari pola 8 digit angka di key
    match = re.search(r'(\d{8})', key)
    if match:
        return match.group(1)
    return None

def extract_datetime_from_key(key):
    """Ekstrak datetime lengkap dari key."""
    # Cari pola 14 digit (YYYYMMDDHHMMSS)
    match = re.search(r'(\d{14})', key)
    if match:
        try:
            return datetime.strptime(match.group(1), '%Y%m%d%H%M%S')
        except:
            pass
    return None

# =============================================
# 4. COCOKKAN DENGAN GEMPA MERUSAK
# =============================================

print("\n📊 Mencocokkan dengan daftar gempa merusak...")
print("-"*70)

found_events = []
not_found_events = []

# Buat set tanggal dari JSON
json_dates = set()
json_datetimes = {}
for key in data.keys():
    date_str = extract_date_from_key(key)
    if date_str:
        json_dates.add(date_str)
        dt = extract_datetime_from_key(key)
        if dt:
            key_date = dt.strftime('%Y%m%d')
            if key_date not in json_datetimes:
                json_datetimes[key_date] = []
            json_datetimes[key_date].append({
                'key': key,
                'datetime': dt,
                'station': key.split('_')[1] if len(key.split('_')) > 1 else 'unknown'
            })

# Cek setiap gempa merusak
for eq in MAJOR_EARTHQUAKES:
    year = eq['year']
    month = eq['month']
    day = eq['day']
    mag = eq['mag']
    name = eq['name']
    
    date_str = f"{year}{month:02d}{day:02d}"
    
    # Perbaikan: cek apakah date_str ada di json_dates dan json_datetimes
    if date_str in json_dates and date_str in json_datetimes and len(json_datetimes[date_str]) > 0:
        events_on_date = json_datetimes.get(date_str, [])
        # Urutkan berdasarkan waktu
        events_on_date.sort(key=lambda x: x['datetime'])
        
        # Ambil yang pertama
        best = events_on_date[0]
        
        found_events.append({
            'name': name,
            'expected_mag': mag,
            'date': date_str,
            'key': best['key'],
            'station': best['station']
        })
    else:
        # Coba cari dengan toleransi ±1 hari
        found = False
        for delta in [-1, 1]:
            check_date = datetime(year, month, day) + timedelta(days=delta)
            check_str = check_date.strftime('%Y%m%d')
            if check_str in json_dates and check_str in json_datetimes and len(json_datetimes[check_str]) > 0:
                events_on_date = json_datetimes.get(check_str, [])
                if events_on_date:
                    best = events_on_date[0]
                    found_events.append({
                        'name': name,
                        'expected_mag': mag,
                        'date': f"{date_str} (≈{check_str})",
                        'key': best['key'],
                        'station': best['station']
                    })
                    found = True
                    break
        
        if not found:
            not_found_events.append({
                'name': name,
                'expected_mag': mag,
                'date': date_str
            })

# =============================================
# 5. TAMPILKAN HASIL
# =============================================

print("\n" + "="*70)
print("📊 HASIL PENCARIAN GEMPA MERUSAK")
print("="*70)

print(f"\n✅ Ditemukan: {len(found_events)} dari {len(MAJOR_EARTHQUAKES)}")
print(f"❌ Tidak ditemukan: {len(not_found_events)}")

# Tampilkan yang ditemukan
if found_events:
    print("\n✅ GEMPA MERUSAK YANG DITEMUKAN:")
    print("-"*70)
    for eq in found_events:
        print(f"  {eq['name']:20} | Mag: {eq['expected_mag']:.1f} | {eq['key']}")
        print(f"    Station: {eq['station']}")

# Tampilkan yang tidak ditemukan
if not_found_events:
    print("\n❌ GEMPA MERUSAK YANG TIDAK DITEMUKAN:")
    print("-"*70)
    for eq in not_found_events:
        print(f"  {eq['name']:20} | Mag: {eq['expected_mag']:.1f} | Tanggal: {eq['date']}")

# =============================================
# 6. STATISTIK
# =============================================

print("\n" + "="*70)
print("📊 STATISTIK")
print("="*70)

# Hitung distribusi tahun dari JSON
years = set()
for key in data.keys():
    date_str = extract_date_from_key(key)
    if date_str and len(date_str) >= 4:
        years.add(date_str[:4])

print(f"Tahun dalam JSON: {sorted(years)}")

# Cek tahun mana yang memiliki gempa merusak
major_years = set([str(eq['year']) for eq in MAJOR_EARTHQUAKES])
missing_years = major_years - years

print(f"Tahun gempa merusak yang tidak ada di JSON: {sorted(missing_years)}")

# =============================================
# 7. RINGKASAN
# =============================================

print("\n" + "="*70)
print("📊 RINGKASAN")
print("="*70)
print(f"Total event dalam JSON: {len(data)}")
print(f"Gempa merusak dicari: {len(MAJOR_EARTHQUAKES)}")
print(f"Ditemukan: {len(found_events)} ({len(found_events)/len(MAJOR_EARTHQUAKES)*100:.1f}%)")
print(f"Tidak ditemukan: {len(not_found_events)}")
print("="*70)

🔍 MEMERIKSA GEMPA MERUSAK DALAM JSON

📂 Membaca JSON: /Volumes/Extreme SSD/unduhan_waveform_merged/extracted_data.json
✅ Total event dalam JSON: 20164

📊 Mencocokkan dengan daftar gempa merusak...
----------------------------------------------------------------------

📊 HASIL PENCARIAN GEMPA MERUSAK

✅ Ditemukan: 14 dari 18
❌ Tidak ditemukan: 4

✅ GEMPA MERUSAK YANG DITEMUKAN:
----------------------------------------------------------------------
  Aceh 2004            | Mag: 9.1 | 2004_GE_UGM_official20041226005853450_30_Aceh
    Station: GE
  Nias 2005            | Mag: 8.6 | 2005_GE_UGM_official20050328160936530_30_Nias
    Station: GE
  Bengkulu 2007        | Mag: 8.5 | 2007_GE_MNAI_official20070912111026830_34_Bengkulu
    Station: GE
  Padang 2009          | Mag: 7.6 | 2009_GE_BKNI_BMKG-20090930101610-001_Padang
    Station: GE
  Sumatra 2010         | Mag: 7.7 | 2010_GE_GSI_BMKG-20100406221503-001_Sumatra
    Station: GE
  Mentawai 2010        | Mag: 7.7 | 2010_GE_MNAI_BMKG-2010

In [5]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
INVESTIGASI GEMPA MERUSAK YANG TIDAK DITEMUKAN
Mencari file .mseed untuk gempa yang hilang di direktori utama dan major.
"""

import os
import re
import json
from pathlib import Path
from datetime import datetime
import pandas as pd

# =============================================
# 1. KONFIGURASI
# =============================================

DIR_MAIN = "/Volumes/Extreme SSD/unduhan_waveform_geofon_juli"
DIR_MAJOR = "/Volumes/Extreme SSD/unduhan_waveform_major_earthquakes"
JSON_PATH = "/Volumes/Extreme SSD/unduhan_waveform_merged/extracted_data.json"

# Daftar gempa yang hilang (dari hasil sebelumnya)
MISSING_EARTHQUAKES = [
    {"year": 2006, "month": 5, "day": 27, "mag": 6.3, "name": "Yogyakarta 2006", "patterns": ["20060527", "YOGI", "yogya"]},
    {"year": 2006, "month": 7, "day": 17, "mag": 7.7, "name": "Pangandaran 2006", "patterns": ["20060717", "BUM", "pangandaran"]},
    {"year": 2013, "month": 4, "day": 6, "mag": 7.2, "name": "Papua 2013", "patterns": ["20130406", "papua"]},
    {"year": 2018, "month": 9, "day": 28, "mag": 7.5, "name": "Palu 2018", "patterns": ["20180928", "palu", "TOLI2"]},
]

# =============================================
# 2. CARI FILE DI KEDUA DIREKTORI
# =============================================

print("="*70)
print("🔍 INVESTIGASI GEMPA MERUSAK YANG HILANG")
print("="*70)

# Kumpulkan semua file dari kedua direktori
all_files = []
for d in [DIR_MAIN, DIR_MAJOR]:
    if os.path.exists(d):
        files = list(Path(d).glob("*.mseed"))
        files = [f for f in files if not f.name.startswith('._')]
        all_files.extend([(f, d) for f in files])

print(f"✅ Total file ditemukan: {len(all_files)}")

# =============================================
# 3. CARI UNTUK SETIAP GEMPA
# =============================================

print("\n📊 Mencari file untuk gempa yang hilang...")
print("-"*70)

results = []

for eq in MISSING_EARTHQUAKES:
    name = eq['name']
    year = eq['year']
    month = eq['month']
    day = eq['day']
    patterns = eq['patterns']
    date_str = f"{year}{month:02d}{day:02d}"
    
    print(f"\n🔍 Mencari: {name} (M{eq['mag']:.1f}) - {date_str}")
    
    found_files = []
    for file_path, source_dir in all_files:
        file_name = file_path.name
        # Cek apakah file mengandung pola yang dicari
        matched = False
        for pattern in patterns:
            if pattern.lower() in file_name.lower():
                matched = True
                break
        # Cek juga jika tanggal ada di nama file
        if date_str in file_name:
            matched = True
        
        if matched:
            found_files.append({
                'file': file_name,
                'source': 'main' if source_dir == DIR_MAIN else 'major',
                'path': str(file_path)
            })
    
    if found_files:
        print(f"  ✅ Ditemukan {len(found_files)} file:")
        for f in found_files:
            print(f"     - {f['file']} ({f['source']})")
    else:
        print(f"  ❌ Tidak ada file ditemukan di kedua direktori.")
    
    # Cek apakah ada di JSON
    with open(JSON_PATH, 'r') as f:
        json_data = json.load(f)
    
    json_found = False
    for key in json_data.keys():
        if date_str in key:
            json_found = True
            break
    
    if json_found:
        print(f"  ✅ Ada di JSON")
    else:
        print(f"  ❌ Tidak ada di JSON")
    
    results.append({
        'name': name,
        'date': date_str,
        'files_found': len(found_files),
        'file_list': [f['file'] for f in found_files],
        'in_json': json_found
    })

# =============================================
# 4. RINGKASAN
# =============================================

print("\n" + "="*70)
print("📊 RINGKASAN INVESTIGASI")
print("="*70)

for r in results:
    status = "✅" if r['files_found'] > 0 else "❌"
    json_status = "✅" if r['in_json'] else "❌"
    print(f"{status} {r['name']:20} | File: {r['files_found']} | In JSON: {json_status}")
    if r['files_found'] > 0:
        print(f"   File: {', '.join(r['file_list'][:3])}")

print("="*70)

# =============================================
# 5. REKOMENDASI
# =============================================

print("\n💡 REKOMENDASI:")
for r in results:
    if r['files_found'] == 0:
        print(f"  ❌ {r['name']}: Tidak ada file .mseed sama sekali. Mungkin perlu diunduh manual.")
    elif r['files_found'] > 0 and not r['in_json']:
        print(f"  ⚠️ {r['name']}: File ada tetapi tidak di JSON. Jalankan ulang ekstraksi pada file tersebut.")
    elif r['files_found'] > 0 and r['in_json']:
        print(f"  ✅ {r['name']}: Sudah ada di JSON.")

🔍 INVESTIGASI GEMPA MERUSAK YANG HILANG
✅ Total file ditemukan: 20232

📊 Mencari file untuk gempa yang hilang...
----------------------------------------------------------------------

🔍 Mencari: Yogyakarta 2006 (M6.3) - 20060527
  ✅ Ditemukan 395 file:
     - GE_YOGI_20060113_074257.mseed (main)
     - GE_YOGI_20060113_185830.mseed (main)
     - GE_YOGI_20060114_004208.mseed (main)
     - GE_YOGI_20060114_174927.mseed (main)
     - GE_YOGI_20060115_011944.mseed (main)
     - GE_YOGI_20060115_034238.mseed (main)
     - GE_YOGI_20060118_140324.mseed (main)
     - GE_YOGI_20060118_115926.mseed (main)
     - GE_YOGI_20060120_202651.mseed (main)
     - GE_YOGI_20060121_093450.mseed (main)
     - GE_YOGI_20060121_210136.mseed (main)
     - GE_YOGI_20060125_010046.mseed (main)
     - GE_YOGI_20060125_033036.mseed (main)
     - GE_YOGI_20060125_135618.mseed (main)
     - GE_YOGI_20060125_140115.mseed (main)
     - GE_YOGI_20060126_053504.mseed (main)
     - GE_YOGI_20060126_162143.mseed (main

In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
VISUALISASI GEMPA MERUSAK INDONESIA DALAM JSON
Berdasarkan hasil investigasi.
"""

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. DATA HASIL INVESTIGASI
# =============================================

# Data gempa merusak (18 gempa)
earthquakes = [
    {"name": "Aceh 2004", "year": 2004, "mag": 9.1, "status": "found"},
    {"name": "Nias 2005", "year": 2005, "mag": 8.6, "status": "found"},
    {"name": "Yogyakarta 2006", "year": 2006, "mag": 6.3, "status": "found"},
    {"name": "Pangandaran 2006", "year": 2006, "mag": 7.7, "status": "found"},
    {"name": "Bengkulu 2007", "year": 2007, "mag": 8.5, "status": "found"},
    {"name": "Padang 2009", "year": 2009, "mag": 7.6, "status": "found"},
    {"name": "Sumatra 2010", "year": 2010, "mag": 7.7, "status": "found"},
    {"name": "Mentawai 2010", "year": 2010, "mag": 7.7, "status": "found"},
    {"name": "Aceh 2012", "year": 2012, "mag": 8.6, "status": "found"},
    {"name": "Papua 2013", "year": 2013, "mag": 7.2, "status": "not_found"},
    {"name": "Pidie Jaya 2016", "year": 2016, "mag": 6.5, "status": "found"},
    {"name": "Lombok 2018", "year": 2018, "mag": 7.0, "status": "found"},
    {"name": "Palu 2018", "year": 2018, "mag": 7.5, "status": "file_exists"},
    {"name": "Maluku 2019", "year": 2019, "mag": 7.3, "status": "found"},
    {"name": "Mamuju 2021", "year": 2021, "mag": 7.0, "status": "found"},
    {"name": "Cianjur 2022", "year": 2022, "mag": 5.6, "status": "found"},
    {"name": "Maluku 2023", "year": 2023, "mag": 7.2, "status": "found"},
    {"name": "Maluku 2024", "year": 2024, "mag": 7.0, "status": "found"},
]

# Status mapping
status_colors = {
    "found": "#2ecc71",       # hijau
    "not_found": "#e74c3c",   # merah
    "file_exists": "#f39c12"  # orange
}
status_labels = {
    "found": "✅ Ditemukan di JSON",
    "not_found": "❌ Tidak ditemukan",
    "file_exists": "⚠️ File ada, gagal ekstrak"
}

df = pd.DataFrame(earthquakes)

# =============================================
# 2. VISUALISASI
# =============================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Gempa Merusak Indonesia (2004-2024) - Status di Dataset', 
             fontsize=16, fontweight='bold')

# --- 2a. Status per Gempa (Bar Chart horizontal) ---
ax1 = axes[0, 0]
df_sorted = df.sort_values('year', ascending=False)
colors_bar = [status_colors[s] for s in df_sorted['status']]
bars = ax1.barh(df_sorted['name'], df_sorted['mag'], color=colors_bar, alpha=0.8)
ax1.set_xlabel('Magnitudo')
ax1.set_title('Status Gempa Merusak per Magnitudo')
ax1.axvline(6.0, color='gray', linestyle='--', alpha=0.5, label='M≥6.0')
ax1.legend([plt.Line2D([0],[0], color='gray', linestyle='--')], ['M≥6.0'])
ax1.grid(True, alpha=0.3, axis='x')

# Tambahkan status di ujung bar
for bar, status in zip(bars, df_sorted['status']):
    label = status_labels.get(status, status)
    ax1.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2, 
             label, va='center', fontsize=8)

# --- 2b. Jumlah per Tahun ---
ax2 = axes[0, 1]
yearly_counts = df['year'].value_counts().sort_index()
colors_year = ['#2ecc71' if y in df[df['status']=='found']['year'].values else '#e74c3c' for y in yearly_counts.index]
bars = ax2.bar(yearly_counts.index, yearly_counts.values, color=colors_year, alpha=0.7)
ax2.set_xlabel('Tahun')
ax2.set_ylabel('Jumlah Gempa Merusak')
ax2.set_title('Jumlah Gempa Merusak per Tahun')
ax2.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, yearly_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.05, str(val), ha='center', va='bottom')

# --- 2c. Distribusi Magnitudo (Histogram) ---
ax3 = axes[1, 0]
bins = np.arange(5.0, 9.5, 0.5)
ax3.hist(df['mag'], bins=bins, color='steelblue', alpha=0.7, edgecolor='black')
ax3.axvline(df['mag'].mean(), color='red', linestyle='--', linewidth=2, 
            label=f'Rata-rata: {df["mag"].mean():.2f}')
ax3.set_xlabel('Magnitudo')
ax3.set_ylabel('Frekuensi')
ax3.set_title('Distribusi Magnitudo Gempa Merusak')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# --- 2d. Ringkasan Tabel ---
ax4 = axes[1, 1]
ax4.axis('tight')
ax4.axis('off')

# Hitung statistik
total = len(df)
found = len(df[df['status'] == 'found'])
not_found = len(df[df['status'] == 'not_found'])
file_exists = len(df[df['status'] == 'file_exists'])
found_pct = found / total * 100

summary_data = [
    ['Total gempa merusak', f'{total}'],
    ['✅ Ditemukan di JSON', f'{found} ({found_pct:.1f}%)'],
    ['⚠️ File ada, gagal ekstrak', f'{file_exists}'],
    ['❌ Tidak ditemukan', f'{not_found}'],
    ['Magnitudo rata-rata', f'{df["mag"].mean():.2f}'],
    ['Magnitudo tertinggi', f'{df["mag"].max():.1f} (Aceh 2004)'],
    ['Tahun dengan gempa terbanyak', f'{df["year"].mode().iloc[0]} ({len(df[df["year"]==df["year"].mode().iloc[0]])} gempa)'],
]

table = ax4.table(cellText=summary_data, colLabels=['Metrik', 'Nilai'],
                  cellLoc='center', loc='center',
                  colColours=['#4472C4', '#4472C4'])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)
ax4.set_title('Ringkasan Gempa Merusak', fontsize=14, pad=20)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('major_earthquakes_status.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Visualisasi tersimpan: major_earthquakes_status.png")

# =============================================
# 3. CETAK RINGKASAN KE TERMINAL
# =============================================

print("\n" + "="*70)
print("📊 RINGKASAN GEMPA MERUSAK INDONESIA")
print("="*70)
print(f"Total gempa merusak: {total}")
print(f"✅ Ditemukan di JSON: {found} ({found_pct:.1f}%)")
print(f"⚠️ File ada, gagal ekstrak: {file_exists}")
print(f"❌ Tidak ditemukan: {not_found}")
print("="*70)

print("\n📋 DAFTAR GEMPA MERUSAK:")
print("-"*70)
for _, row in df.iterrows():
    status_icon = "✅" if row['status'] == 'found' else "⚠️" if row['status'] == 'file_exists' else "❌"
    status_text = status_labels.get(row['status'], row['status'])
    print(f"  {status_icon} {row['name']:20} | M{row['mag']:.1f} | {row['year']} | {status_text}")
print("="*70)

✅ Visualisasi tersimpan: major_earthquakes_status.png

📊 RINGKASAN GEMPA MERUSAK INDONESIA
Total gempa merusak: 18
✅ Ditemukan di JSON: 16 (88.9%)
⚠️ File ada, gagal ekstrak: 1
❌ Tidak ditemukan: 1

📋 DAFTAR GEMPA MERUSAK:
----------------------------------------------------------------------
  ✅ Aceh 2004            | M9.1 | 2004 | ✅ Ditemukan di JSON
  ✅ Nias 2005            | M8.6 | 2005 | ✅ Ditemukan di JSON
  ✅ Yogyakarta 2006      | M6.3 | 2006 | ✅ Ditemukan di JSON
  ✅ Pangandaran 2006     | M7.7 | 2006 | ✅ Ditemukan di JSON
  ✅ Bengkulu 2007        | M8.5 | 2007 | ✅ Ditemukan di JSON
  ✅ Padang 2009          | M7.6 | 2009 | ✅ Ditemukan di JSON
  ✅ Sumatra 2010         | M7.7 | 2010 | ✅ Ditemukan di JSON
  ✅ Mentawai 2010        | M7.7 | 2010 | ✅ Ditemukan di JSON
  ✅ Aceh 2012            | M8.6 | 2012 | ✅ Ditemukan di JSON
  ❌ Papua 2013           | M7.2 | 2013 | ❌ Tidak ditemukan
  ✅ Pidie Jaya 2016      | M6.5 | 2016 | ✅ Ditemukan di JSON
  ✅ Lombok 2018          | M7.0 | 201

In [12]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
VISUALISASI VALIDASI UNDUHAN WAVEFORM
Membandingkan katalog dengan file yang berhasil diunduh.
"""

import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================

WAVEFORM_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon"
#WAVEFORM_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon_juli"
CATALOG_CSV = "/Volumes/Extreme SSD/unduhan_juni_bmkg_usgs/hasilscan/eda_final_output/HYBRID_EARTHQUAKE_CATALOG_2001_2024_FIX.csv"
OUTPUT_IMG = "unduhan_validation_summary_01.png"

# =============================================
# 2. BACA DATA
# =============================================

print("="*70)
print("📊 VISUALISASI VALIDASI UNDUHAN WAVEFORM")
print("="*70)

# --- 2a. Baca file .mseed ---
print("\n📂 Membaca file .mseed...")
files = list(Path(WAVEFORM_DIR).glob("*.mseed"))
files = [f for f in files if not f.name.startswith('._')]
print(f"✅ Total file .mseed: {len(files)}")

# Ekstrak timestamp dari nama file
downloaded_data = []
for f in files:
    parts = f.stem.split('_')
    if len(parts) >= 3:
        network = parts[0]
        station = parts[1]
        # Cari 8 digit timestamp
        import re
        match = re.search(r'(\d{8})', f.name)
        if match:
            timestamp = match.group(1)
            try:
                year = int(timestamp[:4])
                month = int(timestamp[4:6])
                day = int(timestamp[6:8])
                downloaded_data.append({
                    'file': f.name,
                    'network': network,
                    'station': station,
                    'year': year,
                    'timestamp': timestamp
                })
            except:
                pass

df_downloaded = pd.DataFrame(downloaded_data)
print(f"✅ Metadata diekstrak dari {len(df_downloaded)} file")

# --- 2b. Baca katalog ---
print("\n📂 Membaca katalog...")
df_catalog = pd.read_csv(CATALOG_CSV)
df_catalog['datetime'] = pd.to_datetime(df_catalog['time_utc'], utc=True)
df_catalog['year'] = df_catalog['datetime'].dt.year
df_catalog['date'] = df_catalog['datetime'].dt.date
print(f"✅ Total event di katalog: {len(df_catalog)}")

# =============================================
# 3. STATISTIK PERBANDINGAN
# =============================================

# Hitung distribusi tahun dari katalog (untuk filter yang digunakan)
# Filter yang digunakan saat unduhan: M>=4.5, tahun>=2004
df_catalog_filtered = df_catalog[
    (df_catalog['magnitude'] >= 4.5) &
    (df_catalog['year'] >= 2004)
].copy()
print(f"✅ Katalog setelah filter: {len(df_catalog_filtered)}")

# Distribusi tahun dari file yang diunduh
downloaded_yearly = df_downloaded['year'].value_counts().sort_index()

# Distribusi tahun dari katalog (filtered)
catalog_yearly = df_catalog_filtered['year'].value_counts().sort_index()

# Magnitudo dari file yang diunduh (jika bisa dicocokkan)
# Kita coba match berdasarkan tanggal dengan toleransi 1 hari
downloaded_magnitudes = []
downloaded_lats = []
downloaded_lons = []

for _, row in df_downloaded.iterrows():
    # Cari event di katalog dengan tanggal yang sama
    date_obj = datetime.strptime(str(row['year']) + row['timestamp'][4:8], '%Y%m%d').date()
    candidates = df_catalog_filtered[df_catalog_filtered['date'] == date_obj]
    if len(candidates) > 0:
        best = candidates.iloc[0]
        downloaded_magnitudes.append(best['magnitude'])
        downloaded_lats.append(best['latitude'])
        downloaded_lons.append(best['longitude'])

print(f"✅ Jumlah event dengan magnitudo valid: {len(downloaded_magnitudes)}")

# =============================================
# 4. VISUALISASI
# =============================================

fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 3, height_ratios=[1, 1], hspace=0.3, wspace=0.3)

# --- 4a. Map sebaran event yang diunduh ---
ax1 = fig.add_subplot(gs[0, 0])
if len(downloaded_lats) > 0:
    sc = ax1.scatter(downloaded_lons, downloaded_lats, 
                     c=downloaded_magnitudes, s=3, cmap='viridis', alpha=0.6)
    ax1.set_xlabel('Longitude')
    ax1.set_ylabel('Latitude')
    ax1.set_title(f'Sebaran Event Terunduh (n={len(downloaded_lats)})')
    ax1.set_xlim(90, 145)
    ax1.set_ylim(-12, 8)
    ax1.grid(True, alpha=0.3)
    cbar = plt.colorbar(sc, ax=ax1)
    cbar.set_label('Magnitudo')
else:
    ax1.text(0.5, 0.5, 'Tidak ada data', ha='center', va='center')

# --- 4b. Distribusi tahun (perbandingan) ---
ax2 = fig.add_subplot(gs[0, 1])
# Gabungkan kedua series
all_years = sorted(set(downloaded_yearly.index) | set(catalog_yearly.index))
x = np.arange(len(all_years))
width = 0.35

downloaded_vals = [downloaded_yearly.get(y, 0) for y in all_years]
catalog_vals = [catalog_yearly.get(y, 0) for y in all_years]

ax2.bar(x - width/2, catalog_vals, width, label='Katalog (filtered)', color='steelblue', alpha=0.7)
ax2.bar(x + width/2, downloaded_vals, width, label='Terunduh', color='coral', alpha=0.7)
ax2.set_xticks(x)
ax2.set_xticklabels(all_years, rotation=45)
ax2.set_xlabel('Tahun')
ax2.set_ylabel('Jumlah Event')
ax2.set_title('Distribusi Event per Tahun')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# --- 4c. Statistik ringkasan ---
ax3 = fig.add_subplot(gs[0, 2])
ax3.axis('tight')
ax3.axis('off')

total_catalog = len(df_catalog_filtered)
total_downloaded = len(df_downloaded)
success_rate = total_downloaded / total_catalog * 100 if total_catalog > 0 else 0

# Hitung jumlah stasiun unik
unique_stations = df_downloaded['station'].nunique()

summary_data = [
    ['Event target (filtered)', f'{total_catalog:,}'],
    ['File .mseed terunduh', f'{total_downloaded:,}'],
    ['Success rate', f'{success_rate:.1f}%'],
    ['Stasiun unik', f'{unique_stations}'],
    ['Rentang tahun', f"{all_years[0]} - {all_years[-1]}" if all_years else 'N/A'],
    ['Jaringan terbanyak', df_downloaded['network'].mode().iloc[0] if not df_downloaded.empty else 'N/A'],
]

table = ax3.table(cellText=summary_data, colLabels=['Metrik', 'Nilai'],
                  cellLoc='center', loc='center',
                  colColours=['#4472C4', '#4472C4'])
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 1.8)
ax3.set_title('Ringkasan Unduhan', fontsize=14, pad=20)

# --- 4d. Distribusi stasiun (top 10) ---
ax4 = fig.add_subplot(gs[1, 0])
if not df_downloaded.empty:
    top_stations = df_downloaded['station'].value_counts().head(10)
    bars = ax4.barh(top_stations.index, top_stations.values, color='steelblue', alpha=0.7)
    ax4.set_xlabel('Jumlah File')
    ax4.set_title('10 Stasiun Terbanyak')
    ax4.grid(True, alpha=0.3, axis='x')
    for bar, val in zip(bars, top_stations.values):
        ax4.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2, 
                 f'{val}', va='center', fontsize=9)

# --- 4e. Distribusi magnitudo (perbandingan) ---
ax5 = fig.add_subplot(gs[1, 1])
if len(downloaded_magnitudes) > 0:
    # Katalog magnitudo
    catalog_mags = df_catalog_filtered['magnitude'].dropna()
    bins = np.arange(4.0, 9.5, 0.5)
    ax5.hist(catalog_mags, bins=bins, alpha=0.5, label='Katalog', color='steelblue', edgecolor='black')
    ax5.hist(downloaded_magnitudes, bins=bins, alpha=0.7, label='Terunduh', color='coral', edgecolor='black')
    ax5.set_xlabel('Magnitudo')
    ax5.set_ylabel('Frekuensi')
    ax5.set_title('Distribusi Magnitudo')
    ax5.legend()
    ax5.grid(True, alpha=0.3, axis='y')

# --- 4f. Jaringan distribusi ---
ax6 = fig.add_subplot(gs[1, 2])
if not df_downloaded.empty:
    network_counts = df_downloaded['network'].value_counts()
    # Gabungkan yang kecil
    threshold = 2.0
    others = network_counts[network_counts / len(df_downloaded) * 100 < threshold].sum()
    main_networks = network_counts[network_counts / len(df_downloaded) * 100 >= threshold]
    if others > 0:
        main_networks['Other'] = others
    ax6.pie(main_networks.values, labels=main_networks.index, autopct='%1.1f%%')
    ax6.set_title('Distribusi Jaringan Stasiun')

plt.suptitle('Validasi Unduhan Waveform vs Katalog', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig(OUTPUT_IMG, dpi=300, bbox_inches='tight')
plt.close()
print(f"✅ Grafik tersimpan: {OUTPUT_IMG}")

# =============================================
# 5. RINGKASAN
# =============================================

print("\n" + "="*70)
print("📊 RINGKASAN VALIDASI UNDUHAN")
print("="*70)
print(f"Event target (M≥4.5, tahun≥2004): {total_catalog:,}")
print(f"File .mseed terunduh: {total_downloaded:,}")
print(f"Success rate: {success_rate:.1f}%")
print(f"Stasiun unik: {unique_stations}")
print(f"Jaringan terbanyak: {df_downloaded['network'].mode().iloc[0] if not df_downloaded.empty else 'N/A'}")
print(f"Stasiun terbanyak: {df_downloaded['station'].mode().iloc[0] if not df_downloaded.empty else 'N/A'}")
print("="*70)

📊 VISUALISASI VALIDASI UNDUHAN WAVEFORM

📂 Membaca file .mseed...
✅ Total file .mseed: 12940
✅ Metadata diekstrak dari 12940 file

📂 Membaca katalog...
✅ Total event di katalog: 232301
✅ Katalog setelah filter: 25438
✅ Jumlah event dengan magnitudo valid: 12900
✅ Grafik tersimpan: unduhan_validation_summary_01.png

📊 RINGKASAN VALIDASI UNDUHAN
Event target (M≥4.5, tahun≥2004): 25,438
File .mseed terunduh: 12,940
Success rate: 50.9%
Stasiun unik: 29
Jaringan terbanyak: GE
Stasiun terbanyak: TNTI
